In [1]:
# Sentiment Analysis -- (1) Positive , (2) Negative
# Text Preprocesing 

# Steps :
# Load data -> Text pre-processing -> Tensors , Datasets , DataLoaders -> RNN -> train , evaluation -> new review -> RNN -> (positive or negative)

In [52]:
import pandas as pd

In [53]:
df = pd.read_csv("IMDB Dataset.csv")
df.shape

(50000, 2)

In [54]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [55]:
df.drop_duplicates(inplace = True)

In [56]:
df.shape

(49582, 2)

## Text Preprocessing

In [57]:
import re

# text = "abc is writter , abc"

# new_text = re.sub("abc" , "xyz" , text)

# print(new_text)

### 1. convert to lower case

In [58]:
df["review"] = df["review"].str.lower()

### 2. Remove URL's (https , http)

In [59]:
def remove_URLs(text):
    text = re.sub(r"http\S+" , "" , text)
    return text

df["review"] = df["review"].apply(remove_URLs)

### 3. Remove HTML Tags

In [60]:
def remove_tag(text):
    text = re.sub(r"<.*?>" , "" , text)
    return text

df["review"] = df["review"].apply(remove_tag)

### 4. Remove Punctuations

In [61]:
def remove_punct(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)
    return text

df["review"] = df["review"].apply(remove_punct)

In [62]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Removing Stopwords

In [21]:
pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [regex]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------

In [22]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hemant\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hemant\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hemant\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [63]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [64]:
# sample_text = "I like coding in python"
# word_tokenize(sample_text)

In [65]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stopword = set(stopwords.words("english"))
    # Rebuild the string using only non-stopwords
    filtered_tokens = [w for w in tokens if w not in stopword]
    return " ".join(filtered_tokens)

In [66]:
df["review"] = df["review"].apply(remove_stopwords)

In [67]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 6. Stemming

In [68]:
# running -> run
# played -> play
# Use PorterStemming

from nltk.stem import PorterStemmer

In [69]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [70]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7. Encoding 

In [71]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [72]:
y = df["sentiment"]

In [73]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf =  TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

In [75]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4048128 stored elements and shape (49582, 5000)>

### Dataset & Data Loaders

In [76]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X , y, test_size = 0.2 , random_state = 42
)

In [77]:
X_train.shape

(39665, 5000)

In [78]:
import torch
from torch.utils.data import TensorDataset , DataLoader

In [79]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [80]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [81]:
train_loader = DataLoader(train_set , shuffle = True , batch_size = 64)
test_loader = DataLoader(test_set , shuffle = True , batch_size = 64)

### Build Our RNN

In [82]:
import torch.nn as nn
import torch.optim as optim

In [83]:
class RNN(nn.Module):
    def __init__(self , input_size , hidden_size = 128 , num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN Layers
        self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first = True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size , 1)
    
    def forward(self , x):
        # optional => shape (num of layers , batch_size , hidden_size)
        h0 = torch.zeros(self.num_layers , x.size(0) , self.hidden_size)

        out , _ = self.rnn(x , h0)
        # 1st value = hidden state of all the timesteps => (batch , seq_len , hidden size)
        # 2nd value = final hidden state of last timesteps

        out = self.fc(out[: , -1 , :])
        return out

In [84]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [85]:
epochs = 10
for epoch in range(epochs):
    model.train()

    for Xb , yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size , 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size , ) => probability

        loss = criterion(outputs , yb) # compute loss
        loss.backward() # backward
        optimizer.step() # weight updates

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.29558688402175903
epoch = 2/10 and loss = 0.15280848741531372
epoch = 3/10 and loss = 0.16746531426906586
epoch = 4/10 and loss = 0.15721815824508667
epoch = 5/10 and loss = 0.1844795197248459
epoch = 6/10 and loss = 0.30251482129096985
epoch = 7/10 and loss = 0.2140095978975296
epoch = 8/10 and loss = 0.26081526279449463
epoch = 9/10 and loss = 0.14160895347595215
epoch = 10/10 and loss = 0.15848514437675476


In [86]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb , yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals * 100}")

accuracy = 87.0424523545427


In [93]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 1. Define the LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # 2. Define the Dropout layer (Must be inside __init__)
        self.dropout = nn.Dropout(0.3) 

        # 3. Define the fully connected layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Apply dropout to the output of the last time step
        # This helps with the "Overconfidence" issue you saw (1.00 vs 0.00)
        out = self.dropout(out[:, -1, :]) 
        
        out = self.fc(out)
        return out

In [94]:
# 1. Use the new LSTM class instead of RNN
input_size = X_train.shape[1] # This remains 5000 (your TF-IDF features)
hidden_size = 128             # You can experiment with 64 or 256
num_layers = 2                # Adding a second layer helps learn complex patterns

model = LSTMModel(input_size, hidden_size, num_layers)

# 2. Optimization setup
# BCELoss is fine, but ensure you keep the sigmoid in the training loop
criterion = nn.BCEWithLogitsLoss() 

# Lower learning rate (0.001) often helps LSTMs converge without "exploding"
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [95]:
epochs = 10
model.train() # Move this outside if you aren't changing modes per epoch

for epoch in range(epochs):
    running_loss = 0.0
    
    for Xb, yb in train_loader:
        # 1. Reset Gradients
        optimizer.zero_grad()

        # 2. Reshape for LSTM: (Batch, Seq_Len, Features)
        # Your Xb is currently (64, 5000). We make it (64, 1, 5000)
        Xb = Xb.unsqueeze(1) 

        # 3. Forward Pass
        outputs = model(Xb) # Output shape: (batch_size, 1)

        # 4. Handle Sigmoid and Squeezing carefully
        # We use .view(-1) instead of .squeeze() to ensure it's always a 1D vector
        probs = torch.sigmoid(outputs).view(-1) 

        # 5. Calculate Loss
        loss = criterion(probs, yb)
        
        # 6. Backpropagation
        loss.backward()
        
        # 7. Update Weights
        optimizer.step()
        
        running_loss += loss.item()

    # Average loss for the epoch gives a better picture than just the last batch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Average Loss: {avg_loss:.4f}")

Epoch [1/10] - Average Loss: 0.5881
Epoch [2/10] - Average Loss: 0.5484
Epoch [3/10] - Average Loss: 0.5425
Epoch [4/10] - Average Loss: 0.5407
Epoch [5/10] - Average Loss: 0.5385
Epoch [6/10] - Average Loss: 0.5376
Epoch [7/10] - Average Loss: 0.5372
Epoch [8/10] - Average Loss: 0.5364
Epoch [9/10] - Average Loss: 0.5358
Epoch [10/10] - Average Loss: 0.5354


In [96]:
model.eval()

# Tracking metrics for a better "Real World" picture
correct_vals = 0
tot_vals = 0

with torch.no_grad():
    for Xb, yb in test_loader:
        # 1. Match the input shape used in training
        Xb = Xb.unsqueeze(1) 

        # 2. Forward pass
        outputs = model(Xb)
        
        # 3. Use .view(-1) to safely flatten to 1D regardless of batch size
        # This prevents the "0-dim tensor" crash
        probs = torch.sigmoid(outputs).view(-1)
        
        # 4. Thresholding to get binary classes (0 or 1)
        predicted = (probs > 0.5).float()

        # 5. Accumulate results
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

# Calculate Final Accuracy
final_accuracy = (correct_vals / tot_vals) * 100
print(f"Final Test Accuracy: {final_accuracy:.2f}%")

Final Test Accuracy: 86.73%


In [97]:
def predict_sentiment(text, model, vectorizer):
    model.eval()
    # 1. Apply the same preprocessing as training
    text = text.lower()
    text = remove_URLs(text)
    text = remove_tag(text)
    text = remove_punct(text)
    text = remove_stopwords(text)
    text = stemming(text)
    
    # 2. Vectorize and convert to tensor
    vector = vectorizer.transform([text]).toarray()
    vector_tensor = torch.from_numpy(vector).float().unsqueeze(1)
    
    # 3. Predict
    with torch.no_grad():
        output = model(vector_tensor)
        prob = torch.sigmoid(output).item()
        return "Positive" if prob > 0.5 else "Negative", prob

# Test cases with noise
test_reviews = [
    "The movve was gr8, totaly loved it!",           # Typo/Slang
    "Bad plot... but the acting was okay I guess.", # Ambiguity
    "!!!!! HORRIBLE EXPERIENCE !!!!!",              # Excessive Punctuation
    "I've seen better things in a trash can."        # Sarcastic/Idiomatic
]

for r in test_reviews:
    label, score = predict_sentiment(r, model, tf)
    print(f"Review: {r} \nResult: {label} ({score:.2f})\n")

Review: The movve was gr8, totaly loved it! 
Result: Positive (1.00)

Review: Bad plot... but the acting was okay I guess. 
Result: Negative (0.00)

Review: !!!!! HORRIBLE EXPERIENCE !!!!! 
Result: Negative (0.00)

Review: I've seen better things in a trash can. 
Result: Negative (0.00)



In [98]:
# 1. Define the High-Noise test cases
noisy_reviews = [
    "nvr agn... ths movie ws a total disasterrrr!!! 1/10 do nt recommend 2 nyrone.", # Slang/Typos
    "I thought I would hate it, but actually, it wasn't that bad, honestly?",       # Structural Negation
    "BEST. MOVIE. EVER. (jk it was literal garbage)",                               # Sarcasm/Parentheticals
    "L0v3d th3 v1su4ls but th3 st0ry w4s m1d.",                                     # Leetspeak (Numbers for letters)
    "A masterpiece of boredom and a symphony of disappointment.",                   # Sarcastic Metaphors
    "not good not good not good but not bad either.",                               # Repetition/Ambiguity
    "w0uld r4th3r w4tch gr4ss gr0w th4n th1s tr4sh ag41n.",                         # Extreme Leetspeak/Idiom
    "!!!!!!!!!11111oneoneone IT WAS OK I GUESS but why was it so long???????"      # Internet "Shouting" noise
]

# 2. Run the stress test
print(f"{'REVIEW':<60} | {'PREDICTION':<10} | {'CONFIDENCE'}")
print("-" * 85)

for review in noisy_reviews:
    label, score = predict_sentiment(review, model, tf)
    print(f"{review[:58]:<60} | {label:<10} | {score:.4f}")

REVIEW                                                       | PREDICTION | CONFIDENCE
-------------------------------------------------------------------------------------
nvr agn... ths movie ws a total disasterrrr!!! 1/10 do nt    | Negative   | 0.0000
I thought I would hate it, but actually, it wasn't that ba   | Negative   | 0.0000
BEST. MOVIE. EVER. (jk it was literal garbage)               | Negative   | 0.0000
L0v3d th3 v1su4ls but th3 st0ry w4s m1d.                     | Negative   | 0.0002
A masterpiece of boredom and a symphony of disappointment.   | Negative   | 0.0000
not good not good not good but not bad either.               | Negative   | 0.0000
w0uld r4th3r w4tch gr4ss gr0w th4n th1s tr4sh ag41n.         | Negative   | 0.0002
!!!!!!!!!11111oneoneone IT WAS OK I GUESS but why was it s   | Negative   | 0.0000
